# HAYATE — Colab FastH3
無料T4: 診断とテスト。有料GPU: 条件を満たす場合に4ステップ動画生成を検証します。
このPCでは実行せず、Colabのホスト型ランタイムを使用してください。
全セル実行では大容量モデルを取得しません。生成セルは明示的なフラグで有効にします。

In [ ]:
# 1. GitHubから導入（モデルDLなし）
import google.colab
import sys, subprocess, time
from pathlib import Path
SESSION_STARTED = time.monotonic()  # 接続からこのセルまでの時間は費用セルへ加算
REPO = Path('/content/HAYATE')
BRANCH = 'codex/colab-fast-h3'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/server031x-del/hayate.git',str(REPO)], check=True)
print('HAYATE revision:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
sys.path.insert(0, str(REPO/'colab'))
import runtime
print(runtime.inspect_environment())

In [ ]:
# 2. 無料枠でも実行できるテスト（モデル不要）
subprocess.run([sys.executable,'-m','unittest','-v','test_runtime'],cwd=REPO/'colab',check=True)
import ast, json
for source in (REPO/'colab').rglob('*.py'):
    ast.parse(source.read_text(encoding='utf-8-sig'))
notebook = json.loads((REPO/'colab/HAYATE.ipynb').read_text(encoding='utf-8-sig'))
for cell in notebook['cells']:
    if cell['cell_type'] == 'code':
        ast.parse(''.join(cell['source']))
print('Python + notebook syntax OK')

## 有料GPU用（未実測）
SM80以降、VRAM 20 GiB以上、RAM 30 GiB以上が事前条件です。64 GiB以上のRAMを推奨。
T4/TPUではこのVSA経路を停止します。条件は動作保証ではありません。
[MiniMaxライセンス](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)を確認してください。

In [ ]:
# 3. 対応環境の依存導入
ENABLE_GENERATION = False # @param {type:"boolean"}
if ENABLE_GENERATION:
    runtime.setup()
else:
    print('診断モード: インストール・モデル取得・生成をスキップ')

In [ ]:
# 4. 約39.5 GiBのモデルを取得・SHA256検証
DOWNLOAD_MODELS = False # @param {type:"boolean"}
if ENABLE_GENERATION and DOWNLOAD_MODELS:
    runtime.download_models()
else:
    print('モデルは未取得。ライセンス確認後に両方のフラグを有効化してください。')

In [ ]:
# 5. VM内ワーカーを一度起動。同じモデルを繰り返し利用
if ENABLE_GENERATION:
    if 'session' not in globals():
        session = runtime.Session()
    session.start()

In [ ]:
# 6. 4ステップ生成。このセルをseedを変えて再実行するとウォーム計測できます。
PROMPT = 'A cinematic shot of a red sports car driving along a coastal road, smooth tracking camera, natural daylight. Audio: engine sound and ocean waves.' # @param {type:"string"}
SEED = 20260908 # @param {type:"integer"}
if ENABLE_GENERATION:
    result = session.generate(PROMPT, SEED, width=608, height=352, frames=124)
    print(result)
    from IPython.display import Video, display
    for path in result['files']:
        display(Video(path, embed=True))

In [ ]:
# 7. 1採用動画あたりの費用。実際の購入額/CU・ランタイム表示CU/時を入力。
YEN_PER_CU = 0.0 # @param {type:"number"}
CU_PER_HOUR = 0.0 # @param {type:"number"}
ACCEPTED_VIDEOS = 0 # @param {type:"integer"}
BEFORE_FIRST_CELL_SECONDS = 0.0 # @param {type:"number"}
report = runtime.cost_report(time.monotonic()-SESSION_STARTED+BEFORE_FIRST_CELL_SECONDS, ACCEPTED_VIDEOS, YEN_PER_CU, CU_PER_HOUR)
print(report)
print('料金0入力は不明扱い。初回DL・失敗・待機も含む推定。正確な請求は実CU消費差分で確認。')

In [ ]:
# 8. 出力をダウンロード（VM終了前）
if ENABLE_GENERATION and 'result' in globals():
    from google.colab import files
    for path in result['files']:
        files.download(path)

In [ ]:
# 9. 作業終了時だけ実行。再生成はセル5から。
if 'session' in globals():
    session.close()
print('出力保存後にColabのランタイムを接続解除してください。ワーカー停止だけでは接続課金は止まりません。')